# Pokémon Battle Prediction 2026 — Competitive XGBoost Pipeline
**Sapienza University of Rome — Foundations of Data Science Competition**

This notebook implement the complete pipeline, based on XGBoost to predict the winnner (`player_won` $\in \{0, 1\}$) of Pokémon battles  of the 2nd Generation starting from the first 30 turni osserved (`battle_timeline`).

### key points:
1. **Riconstruction of the state of the battle (`src/battle_features.py`)**:
   - **`move_details` Cache Deduplication**: Filters out voluntary switch-in turns where the dataset parser repeats the Pokémon's last cached move, preventing double-counting of offensive and status moves.
   - **Physical Reconstruction of Player 2 Single-KOs**: Since `p2_pokemon_state.status == "fnt"` only appears on simultaneous double-KOs and on Turn 30 (when a P2 Pokémon faints alone on Turns 1–29, its replacement immediately appears in the end-of-turn snapshot), we reconstruct P2 single-KOs using Gen 2 mechanics (verifying an intact replacement, attack executed by P1, type effectiveness, residual status damage, and *Explosion*).
   - **Within-Fold Internal Species Catalogue**: Built exclusively from training data (`p1_team_details` and `p2_lead_details`, 247 species with 0 inconsistencies) to infer base stats and typings of Pokémon revealed by P2.
3. **Six Modular Feature Families + Leak-Free Within-Fold Target Encoding**:
   - **Family A (`team`)**: Base stats of P1's full team, physical/special bulk, speed tiers, type diversity, and Gen 2 OU metagame staples.
   - **Family B (`p2_observed`)**: Stats of P2's lead and revealed Pokémon, uncertainty over unrevealed reserves, and observed move categories.
   - **Family C (`state`)**: Material balance at Turn 30, alive/fainted differential, recency-weighted HP, status severity (distinguishing *Rest* from opponent-inflicted sleep), active boosts, and phazed-absence score.
   - **Family D (`temporal`)**: Multi-window trajectories (`Turns 1–10`, `11–20`, `21–30`, `Last 5`, `Last 10`), linear slope (momentum), lead changes, and pace control.
   - **Family E (`matchup`)**: Gen 2 type effectiveness matrix (`type_chart_gen2.csv`), active STAB multipliers, best available counter on the roster, and Speed/KO threat index.
   - **Family F (`interactions`)**: Non-linear ratios, effective HP-adjusted survival advantage, and Turn 30 *Endgame Sweeper* threat index.
   - **`WithinFoldTargetEncoder`**: Computes smoothed empirical win rates (species, lead matchups, and HP-weighted Turn 30 survivors) strictly within each fold via nested 5-fold OOF.
4. **Automatic Generation of the Single `submission.csv` File**: At the end of the notebook, the single **`submission.csv`** file is generated and validated, ready for automatic submission to the Kaggle challenge.

In [1]:
# Standard library imports for filesystem navigation and process execution
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO_URL = os.environ.get("REPO_URL", "https://github.com/danielegabriele17-rgb/predictive-spikes.git")
GIT_REF = os.environ.get("GIT_REF", "main")
CLONE_DIR = Path("/tmp/predictive-spikes") if Path("/kaggle/working").exists() else Path.cwd()

def setup_repository() -> Path:
    """Locate the local repository root or clone it on Kaggle if not already present."""
    for candidate in [Path.cwd(), Path.cwd().parent, CLONE_DIR]:
        if (candidate / "src" / "battle_features.py").exists() and (candidate / "configs" / "experiments.yaml").exists():
            print(f"Repository rilevato in: {candidate.resolve()}")
            return candidate.resolve()

    print(f"Clonazione di {REPO_URL} (ref={GIT_REF}) in {CLONE_DIR}...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_REF, REPO_URL, str(CLONE_DIR)], check=True)
    return CLONE_DIR.resolve()

# Ensure the project root is in sys.path so `src.*` modules can be imported directly
REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository Root : {REPO_ROOT}")
print(f"Working Dir     : {Path.cwd().resolve()}")

Repository rilevato in: /Users/Daniele/Documents/GitHub/predictive-spikes
Repository Root : /Users/Daniele/Documents/GitHub/predictive-spikes
Working Dir     : /Users/Daniele/Documents/GitHub/predictive-spikes/notebooks


## Import of Pipeline's modules (`src/`)

In [2]:
# Core scientific computing and gradient boosting libraries
import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score

try:
    from IPython.display import display
except ImportError:
    display = print

# Project pipeline modules for feature extraction, CV validation, and submission generation
from src.battle_features import build_species_catalogue
from src.data_loader import discover_data_dir, load_competition_data
from src.feature_engineering import (
    ALL_FEATURE_FAMILIES,
    WithinFoldTargetEncoder,
    build_feature_dataframe,
)
from src.model import get_feature_importance_df, run_xgb_cv
from src.submission import generate_submission, validate_submission_df
from src.validation import get_cv_splits

print(f"XGBoost version: {xgb.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")

XGBoost version: 3.4.1
NumPy version  : 2.5.4
Pandas version : 3.1.0


## Automatic Detection and Dataset Loading


In [3]:
# Automatically discover the dataset directory (local repo or /kaggle/input) and load all files
t_start = time.perf_counter()
DATA_DIR = discover_data_dir()
sample_submission_path = DATA_DIR / "sample_submission.csv"

# Load 10,000 training battles, 5,000 test battles, sample_submission.csv, and Gen 2 type chart
train_battles, test_battles, sample_sub, type_chart = load_competition_data(custom_dir=DATA_DIR)
y = np.array([int(b["player_won"]) for b in train_battles], dtype=int)

print(f"Directory dati attiva : {DATA_DIR}")
print(f"Battaglie di Training : {len(train_battles):,} | Target positive rate: {y.mean():.4f}")
print(f"Battaglie di Test     : {len(test_battles):,} | Sample submission shape: {sample_sub.shape}")
print(f"Gen 2 Type Chart      : {len(type_chart)} coppie attacco-difesa caricate")

Directory dati attiva : /Users/Daniele/Documents/GitHub/predictive-spikes/fds-pokemon-battles-prediction-2026
Battaglie di Training : 10,000 | Target positive rate: 0.5000
Battaglie di Test     : 5,000 | Sample submission shape: (5000, 2)
Gen 2 Type Chart      : 110 coppie attacco-difesa caricate


## Extraction of the 6 Feature Families and Pokédex


In [4]:
# Build internal species catalogue strictly from training data (zero test set leakage)
t_fe = time.perf_counter()
catalogue = build_species_catalogue(train_battles)
print(f"Catalogo interno delle specie costruito dal training set: {len(catalogue)} specie Pokémon registrate")

# Extract all 6 observation-only feature families (A-F) into pandas DataFrames
X_train_full, train_ids, y_extracted = build_feature_dataframe(
    train_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
X_test_full, test_ids, _ = build_feature_dataframe(
    test_battles,
    families=ALL_FEATURE_FAMILIES,
    catalogue=catalogue,
    type_chart=type_chart,
)
print(f"Matrici di feature estratte in {time.perf_counter() - t_fe:.2f}s:")
print(f"  X_train_full: {X_train_full.shape}")
print(f"  X_test_full : {X_test_full.shape}")

Catalogo interno delle specie costruito dal training set: 239 specie Pokémon registrate
Matrici di feature estratte in 8.85s:
  X_train_full: (10000, 549)
  X_test_full : (5000, 549)


## Loading Optimized Configurations(`configs/experiments.yaml`)


In [5]:
# Load version-controlled feature subsets, tuned XGBoost hyperparameters, and ensemble weights
config_path = REPO_ROOT / "configs" / "experiments.yaml"
with open(config_path, "r", encoding="utf-8") as f:
    exp_config = yaml.safe_load(f)

RANDOM_SEED = int(exp_config.get("random_seed", 42))
FINAL_FOLDS = int(exp_config.get("n_splits", 5))
DECISION_THRESHOLD = float(exp_config.get("default_threshold", 0.5))
feature_subsets = exp_config["feature_subsets"]
models_cfg = exp_config["models"]
ens_cfg = exp_config["ensemble"]

print(f"Configurazione caricata da: {config_path}")
for subset_name, cols in feature_subsets.items():
    print(f"  Feature subset [{subset_name:18s}]: {len(cols)} feature")
print("Modelli configurati:", list(models_cfg.keys()))

Configurazione caricata da: /Users/Daniele/Documents/GitHub/predictive-spikes/configs/experiments.yaml
  Feature subset [pruned_features   ]: 511 feature
  Feature subset [top200_features   ]: 200 feature
  Feature subset [top220_features   ]: 220 feature
  Feature subset [top140_features   ]: 140 feature
Modelli configurati: ['model1_optuna_top200_te', 'model2_conservative_depth4_pruned_te', 'model3_obs_only_top220_no_te', 'model4_compact_top140_te']


## 5-Fold Stratified CV Training and Multi-Model Ensemble Blending
We train the XGBoost models across the 5 stratified folds (`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`), computing `WithinFoldTargetEncoder` strictly inside each fold (zero data leakage), and blend the OOF and Test probabilities into the final Ensemble:
- **`model1_optuna_top200_te`** (`weight = 0.15`): Optuna TPE model (`max_depth=4`, `min_child_weight=20`) on the Top-200 features + Within-Fold TE.
- **`model2_conservative_depth4_pruned_te`** (`weight = 0.35`): Heavily regularized shallow model (`max_depth=4`, `colsample_bytree=0.45`, `reg_lambda=5.0`) on all 512 pruned features + Within-Fold TE.
- **`model3_obs_only_top220_no_te`** (`weight = 0.10`): *Pure Observation-Only* model (Zero Target Encoding, `max_depth=5`) on the Top-220 features.
- **`model4_compact_top140_te`** (`weight = 0.40`): Compact high signal-to-noise ratio model (`max_depth=5`, `colsample_bytree=0.60`) on the Top-140 features + Within-Fold TE.

In [6]:
# 1. Generate reproducible 5-fold StratifiedKFold splits
cv_splits = get_cv_splits(train_battles, y, n_splits=FINAL_FOLDS, seed=RANDOM_SEED, group_duplicates=False)

# 2. Precompute strictly within-fold out-of-fold (OOF) Bayesian target encodings (zero leakage)
te_folds_train = []
te_folds_val = []
te_folds_test = []
for tr_idx, va_idx in cv_splits:
    enc = WithinFoldTargetEncoder(smoothing=30.0, team_smoothing=20.0)
    tr_b = [train_battles[i] for i in tr_idx]
    va_b = [train_battles[i] for i in va_idx]
    te_folds_train.append(enc.fit_transform_inner_oof(tr_b, y[tr_idx], n_inner_splits=5, seed=RANDOM_SEED))
    te_folds_val.append(enc.transform(va_b))
    te_folds_test.append(enc.transform(test_battles))

fold_map = {(int(tr[0]), int(va[0])): idx for idx, (tr, va) in enumerate(cv_splits)}

def fold_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_train[idx], te_folds_val[idx]

def test_te_fn(tr_idx: np.ndarray, va_idx: np.ndarray):
    idx = fold_map[(int(tr_idx[0]), int(va_idx[0]))]
    return te_folds_test[idx]

# 3. Train and evaluate each of the 4 diverse XGBoost configurations across the 5 folds
oof_probs_map = {}
test_probs_map = {}
cv_results_map = {}

for m_name, m_spec in models_cfg.items():
    cols = feature_subsets[m_spec["feature_subset"]]
    use_te = bool(m_spec.get("use_within_fold_te", True))
    es_rounds = m_spec.get("early_stopping_rounds", 50)
    params = m_spec["xgb_params"]

    cv_res, test_probs = run_xgb_cv(
        X_df=X_train_full[cols],
        y=y,
        splits=cv_splits,
        xgb_params=params,
        experiment_id=m_name,
        early_stopping_rounds=es_rounds,
        fold_feature_fn=fold_te_fn if use_te else None,
        X_test_df=X_test_full[cols],
        test_fold_feature_fn=test_te_fn if use_te else None,
        threshold=DECISION_THRESHOLD,
        seed=RANDOM_SEED,
    )
    cv_results_map[m_name] = cv_res
    oof_probs_map[m_name] = cv_res.oof_probs
    test_probs_map[m_name] = test_probs
    print(
        f"[{m_name:36s}] 5-Fold CV Acc: {cv_res.cv_mean_acc:.4f} +/- {cv_res.cv_std_acc:.4f} "
        f"| ROC-AUC: {cv_res.oof_roc_auc:.4f} | LogLoss: {cv_res.oof_log_loss:.4f} "
        f"| Train-Val Gap: {np.mean(cv_res.fold_train_accs) - cv_res.cv_mean_acc:.4f}"
    )

# 4. Blend out-of-fold and test-set probabilities using the configured ensemble weights
members = ens_cfg["members"]
weights = np.asarray(ens_cfg["weights"], dtype=float)
weights = weights / np.sum(weights)

blended_oof_probs = sum(w * oof_probs_map[m] for w, m in zip(weights, members))
blended_test_probs = sum(w * test_probs_map[m] for w, m in zip(weights, members))

ens_fold_accs = [
    float(accuracy_score(y[va_idx], (blended_oof_probs[va_idx] >= DECISION_THRESHOLD).astype(int)))
    for _, va_idx in cv_splits
]
final_oof_acc = float(np.mean(ens_fold_accs))
final_oof_std = float(np.std(ens_fold_accs))
final_oof_auc = float(roc_auc_score(y, blended_oof_probs))
final_oof_loss = float(log_loss(y, blended_oof_probs))

print("=" * 80)
print(f"RISULTATO FINALE ENSEMBLE ({FINAL_FOLDS}-Fold x {len(members)} Configurazioni = {FINAL_FOLDS * len(members)} modelli XGBoost)")
print(f"OOF Validation Accuracy : {final_oof_acc:.4f} +/- {final_oof_std:.4f}")
print(f"OOF ROC-AUC             : {final_oof_auc:.4f}")
print(f"OOF Log-Loss            : {final_oof_loss:.4f}")
print("=" * 80)

[model1_optuna_top200_te             ] 5-Fold CV Acc: 0.6970 +/- 0.0022 | ROC-AUC: 0.7738 | LogLoss: 0.5662 | Train-Val Gap: 0.0916
[model2_conservative_depth4_pruned_te] 5-Fold CV Acc: 0.7008 +/- 0.0024 | ROC-AUC: 0.7748 | LogLoss: 0.5657 | Train-Val Gap: 0.1072
[model3_obs_only_top220_no_te        ] 5-Fold CV Acc: 0.6988 +/- 0.0046 | ROC-AUC: 0.7714 | LogLoss: 0.5692 | Train-Val Gap: 0.1453
[model4_compact_top140_te            ] 5-Fold CV Acc: 0.7020 +/- 0.0036 | ROC-AUC: 0.7752 | LogLoss: 0.5647 | Train-Val Gap: 0.1286
RISULTATO FINALE ENSEMBLE (5-Fold x 4 Configurazioni = 20 modelli XGBoost)
OOF Validation Accuracy : 0.7030 +/- 0.0017
OOF ROC-AUC             : 0.7767
OOF Log-Loss            : 0.5636


## Why We Directly Generate a Single `submission.csv` (Final Model Selection)

During the experimental phase, we tested and compared in **5-Fold Stratified Cross-Validation** the separate export of individual candidate models (previously referred to as *Submission 1* based on the single `model4_compact_top140_te` and *Submission 2* based on the single `model2_conservative_depth4_pruned_te`) against the **Multi-Model Diverse XGBoost Ensemble**:

| Tested Candidate | 5-Fold OOF Accuracy | Fold Std Dev | OOF ROC-AUC | OOF Log-Loss | Outcome |
| :--- | :---: | :---: | :---: | :---: | :--- |
| `model1_optuna_top200_te` (Single) | `0.6970` | `± 0.0022` | `0.7738` | `0.5662` | Discarded as single submission |
| `model3_obs_only_top220_no_te` (Single, No TE) | `0.6988` | `± 0.0046` | `0.7714` | `0.5692` | Discarded as single submission |
| `model2_conservative_depth4_pruned_te` (*formerly Submission 2*) | `0.7008` | `± 0.0024` | `0.7748` | `0.5657` | Outperformed by Ensemble |
| `model4_compact_top140_te` (*formerly Submission 1*) | `0.7020` | `± 0.0036` | `0.7752` | `0.5647` | Outperformed by Ensemble |
| **Diverse 4-Model XGBoost Ensemble (`submission.csv`)** | **`0.7030`** | **`± 0.0017`** | **`0.7767`** | **`0.5636`** | **Selected as Single Output** |

### Technical Rationale for the Single Selection
1. **Superiority Across All OOF Metrics**: The weighted ensemble (`0.15 · M1 + 0.35 · M2 + 0.10 · M3 + 0.40 · M4`) consistently outperforms every individual model in **Accuracy (`0.7030`)**, **ROC-AUC (`0.7767`)**, and **Log-Loss (`0.5636`)**.
2. **Halving Cross-Fold Variance (`± 0.0017`)**: Combining trees of varying depths (`max_depth=4` vs `5`), distinct feature subsets (`140`, `200`, `220`, `511`), and the presence/absence of Target Encoding stabilizes predicted probabilities on uncertain battles near the `0.50` threshold, guarding against overfitting on the Private Leaderboard.
3. **Direct Pipeline Without Redundant Files**: By removing the intermediate generation of `submission_1` and `submission_2`, the pipeline directly outputs the single **`submission.csv`** file in the root/working directory, ensuring immediate automatic submission to the Kaggle competition.

In [7]:
# ==============================================================================
# 1. TOP 25 FEATURE IMPORTANCE & DIAGNOSTIC PLOTS (via native Booster.get_score & xgb.plot_importance)
# ==============================================================================
cv_m4 = cv_results_map["model4_compact_top140_te"]
fi_df = get_feature_importance_df(cv_m4.models, cv_m4.feature_names)
print("Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):")
print(fi_df.head(25).to_string(index=False))

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    top_n = min(25, len(fi_df))
    xgb.plot_importance(
        cv_m4.models[0].get_booster(),
        ax=axes[0],
        max_num_features=top_n,
        importance_type="gain",
        title=f"Top {top_n} Feature per Gain (xgb.plot_importance)",
        xlabel="Gain",
        show_values=False,
        grid=False,
    )

    fold_ids = [f"Fold {i}" for i in range(1, FINAL_FOLDS + 1)]
    axes[1].bar(fold_ids, ens_fold_accs, color="#2e8b57", alpha=0.85, label="Fold Val Acc")
    axes[1].axhline(final_oof_acc, color="#d9534f", linestyle="--", linewidth=2, label=f"Blended OOF Acc ({final_oof_acc:.4f})")
    axes[1].set_ylim(min(ens_fold_accs) - 0.03, max(ens_fold_accs) + 0.03)
    axes[1].set_title(f"Accuratezza di Validazione sui {FINAL_FOLDS} Fold")
    axes[1].set_ylabel("Accuracy")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(loc="best")

    plt.tight_layout()
    plt.show()
except ImportError:
    pass

# ==============================================================================
# 2. GENERATE & VALIDATE THE SINGLE KAGGLE SUBMISSION FILE (submission.csv)
#    Uses `src.submission.generate_submission` and `validate_submission_df`
# ==============================================================================
binary_test_preds = (blended_test_probs >= DECISION_THRESHOLD).astype(int)
submission_df = generate_submission(
    test_battle_ids=test_ids,
    test_predictions=binary_test_preds,
    sample_sub_df=sample_sub,
    output_path="submission.csv",
)
sub_info = validate_submission_df(submission_df, sample_sub)

print("=" * 72)
print("File 'submission.csv' generato e validato con successo!")
print(f"Totale battaglie predette : {sub_info['n_rows']}")
print(f"Vittorie Player 1 (1)     : {sub_info['positive_count']} ({sub_info['positive_rate']*100:.2f}%)")
print(f"Vittorie Player 2 (0)     : {sub_info['negative_count']} ({(1 - sub_info['positive_rate'])*100:.2f}%)")
print("=" * 72)

display(submission_df.head(10))

Top 25 Feature per Importance (Multi-Metric Native XGBoost Booster Scores):
                      feature  importance      gain  total_gain  weight      cover
inter_combined_material_score    0.065198 82.416141 8356.507227   101.2 750.667908
          state_hp_ratio_soft    0.031119 39.396878 2715.749219    68.8 558.129852
           state_hp_diff_soft    0.029498 37.455118 2065.520935    54.2 489.537823
           state_hp_ratio_adj    0.022312 28.418891  835.476611    28.8 380.777069
          temp_hp_adv_adj_t30    0.022255 28.117196  704.529089    24.8 367.978925
   temp_last5_hp_adv_adj_mean    0.019759 25.276037  946.713818    37.2 338.171478
            state_hp_diff_raw    0.013343 16.856589  579.869394    33.4 375.403711
          state_clean_hp_diff    0.013288 16.856466 1666.961938    99.8 449.549591
            state_spikes_diff    0.011726 14.812617  969.567639    65.4 592.310895
      inter_hp_raw_x_fnt_diff    0.011374 14.406060  566.256744    39.6 283.359573
  inter_sta

,battle_id,player_won
0,0,1
1,1,1
2,2,0
3,3,0
4,4,1
5,5,0
6,6,1
7,7,0
8,8,0
9,9,0
